# 最小覆盖子串：从计数建模到可变长滑动窗口

这份 notebook 不直接给完整题解，而是带你把题目一步步建模出来：

- 把“包含 t 中每个字符，包括重复字符”翻译成计数约束。
- 先写出正确但慢的区间枚举，观察哪里可以复用。
- 推出可变长滑动窗口的两个动作：右端扩张、左端收缩。
- 用 `missing` 或 `formed/required` 这样的状态，把“窗口是否达标”变成 O(1) 判断。

遇到 `TODO` 时先自己补，后面的检查会帮你确认思路是否完整。

## 0. 题目建模：字符串不是集合，而是多重集合

如果 `t = "ABC"`，窗口里有 `A/B/C` 各一个就够。

但如果 `t = "AABC"`，窗口里必须有：

$$
A \ge 2,\quad B \ge 1,\quad C \ge 1
$$

所以 `t` 应该被建模成一个**字符计数表**，也就是多重集合：

$$
need[c] = c \text{ 在 } t \text{ 中出现的次数}
$$

某个窗口 `s[l:r+1]` 达标，当且仅当：

$$
\forall c,\ window[c] \ge need[c]
$$

In [1]:
from collections import Counter

s = "ADOBECODEBANC"
t = "ABC"

need = Counter(t)
need

Counter({'A': 1, 'B': 1, 'C': 1})

In [2]:
def covers(window_count, need):
    """判断当前窗口计数是否覆盖 t 的计数需求。"""
    return all(window_count[c] >= need[c] for c in need)

for sub in ["ADO", "ADOBEC", "BANC", "BAN"]:
    print(f"{sub!r:8} count={dict(Counter(sub))!s:32} covers={covers(Counter(sub), need)}")

'ADO'    count={'A': 1, 'D': 1, 'O': 1}         covers=False
'ADOBEC' count={'A': 1, 'D': 1, 'O': 1, 'B': 1, 'E': 1, 'C': 1} covers=True
'BANC'   count={'B': 1, 'A': 1, 'N': 1, 'C': 1} covers=True
'BAN'    count={'B': 1, 'A': 1, 'N': 1}         covers=False


## 1. 先写暴力模型：所有子串里找最短达标者

暴力思路是：枚举所有左端点 `l` 和右端点 `r`，检查 `s[l:r+1]` 是否覆盖 `t`。

这个模型很有价值，因为它把目标说清楚了：

$$
\min (r-l+1) \quad \text{s.t.}\quad count(s[l:r+1]) \ge count(t)
$$

但如果每个子串都重新统计字符，复杂度会很高。即使边枚举边维护计数，也会接近 `O(m^2 * alphabet)`。

In [ ]:
def brute_force_min_window(s, t):
    """正确但慢：用来做小规模对照。"""
    need = Counter(t)
    best = None
    for left in range(len(s)):
        window = Counter()
        for right in range(left, len(s)):
            window[s[right]] += 1
            if covers(window, need):
                candidate = s[left:right + 1]
                if best is None or len(candidate) < len(best):
                    best = candidate
                break
    return best or ""

brute_force_min_window(s, t)

## 2. 关键单调性：右端点只会让窗口“更容易达标”

固定左端点 `left` 时，随着 `right` 向右移动，窗口只会增加字符。

一旦 `s[left:right+1]` 已经覆盖 `t`，继续增加右端点仍然覆盖 `t`，只是更长。

这给了我们一个重要启发：

- 右端点负责扩张，直到窗口达标。
- 达标后，左端点负责收缩，直到再收缩就不达标。
- 每个端点都只往右走，不需要回头。

这就是本题从暴力枚举变成 `O(m+n)` 的根本原因。

In [ ]:
def show_fixed_left(s, t, left):
    need = Counter(t)
    window = Counter()
    for right in range(left, len(s)):
        window[s[right]] += 1
        sub = s[left:right + 1]
        print(f"[{left}, {right}] {sub!r:14} covers={covers(window, need)}")

show_fixed_left(s, t, left=0)

## 3. 窗口状态：不要每次都扫描整个 need

如果每次判断 `covers(window, need)` 都遍历 `need`，虽然英文字母规模很小，但从建模角度看还不够干净。

更好的做法是维护一个标量状态。常见有两种等价写法：

1. `formed / required`：有多少种字符已经满足需求。
2. `missing`：还缺多少个必要字符，按重复次数计。

这份练习使用 `missing`，因为它能直观表达“还差几个字符”。

初始化：

$$
missing = |t|
$$

当加入字符 `c` 时，如果加入前 `window[c] < need[c]`，说明这个字符补上了一个缺口，`missing -= 1`。

当移出字符 `c` 时，如果移出前 `window[c] <= need[c]`，说明拿走了一个必要字符，`missing += 1`。

In [ ]:
def trace_missing_on_expand(s, t):
    need = Counter(t)
    window = Counter()
    missing = len(t)
    for right, c in enumerate(s):
        before = window[c]
        if before < need[c]:
            missing -= 1
        window[c] += 1
        print(f"add {c!r} at {right:2}: before={before}, need={need[c]}, missing={missing}")
        if missing == 0:
            print("  first valid prefix:", s[:right + 1])
            break

trace_missing_on_expand(s, t)

## 4. 先写两个原子操作

不要一上来写完整函数。先把“加入一个右端字符”和“移出一个左端字符”写对。

它们共同维护三个对象：

- `need`：目标需求，不变。
- `window`：当前窗口计数，会变。
- `missing`：当前还缺多少个必要字符，会变。

窗口达标的条件就是：

```python
missing == 0
```

In [ ]:
def add_right_char(window, need, missing, c):
    """把字符 c 加入窗口右端，返回更新后的 missing。"""
    # TODO 1:
    # 如果加入前 window[c] < need[c]，说明 c 填补了一个缺口。
    # 然后 window[c] 增加 1。
    # 最后返回 missing。
    return missing


def remove_left_char(window, need, missing, c):
    """把字符 c 从窗口左端移出，返回更新后的 missing。"""
    # TODO 2:
    # 如果移出前 window[c] <= need[c]，说明拿走的是必要字符。
    # 然后 window[c] 减少 1。
    # 最后返回 missing。
    return missing

### 自测 1：原子操作

填好上面的两个函数后，下面应该通过。这里特意测试了重复字符：`t = "AA"`。

In [ ]:
need_aa = Counter("AA")
window_aa = Counter()
missing = len("AA")

missing = add_right_char(window_aa, need_aa, missing, "A")
assert missing == 1 and window_aa["A"] == 1

missing = add_right_char(window_aa, need_aa, missing, "A")
assert missing == 0 and window_aa["A"] == 2

missing = add_right_char(window_aa, need_aa, missing, "A")
assert missing == 0 and window_aa["A"] == 3

missing = remove_left_char(window_aa, need_aa, missing, "A")
assert missing == 0 and window_aa["A"] == 2

missing = remove_left_char(window_aa, need_aa, missing, "A")
assert missing == 1 and window_aa["A"] == 1

"atomic operations passed"

## 5. 双指针框架：扩张直到达标，达标后尽量收缩

现在可以把两个原子操作组合起来。

核心循环结构是：

```text
for right in [0..len(s)-1]:
    把 s[right] 加入窗口

    while 当前窗口已经覆盖 t:
        用当前窗口更新最优答案
        移出 s[left]
        left += 1
```

这里的 `while` 很关键：它不是只收缩一步，而是要一直收缩到窗口刚好不达标。这样每次记录到的候选窗口，都尽量短。

In [ ]:
def guided_min_window(s, t):
    """练习版：依赖你填好的 add/remove 原子操作。"""
    if not s or not t or len(t) > len(s):
        return ""

    need = Counter(t)
    window = Counter()
    missing = len(t)
    left = 0
    best_left = 0
    best_len = float("inf")

    for right, c in enumerate(s):
        # TODO 3: 加入右端字符，并更新 missing。
        # missing = ...

        # TODO 4: 当 missing == 0 时，当前窗口已经覆盖 t。
        # 这时持续尝试收缩左端：
        #   1. 如果当前窗口更短，更新 best_left 和 best_len。
        #   2. 移出 s[left]，更新 missing。
        #   3. left 向右移动一格。
        # while ...:
        #     ...
        pass

    if best_len == float("inf"):
        return ""
    return s[best_left:best_left + best_len]

### 自测 2：题目样例和边界

填好 `guided_min_window` 后，下面应该通过。

In [ ]:
assert guided_min_window("ADOBECODEBANC", "ABC") == "BANC"
assert guided_min_window("a", "a") == "a"
assert guided_min_window("a", "aa") == ""
assert guided_min_window("aa", "aa") == "aa"
assert guided_min_window("bba", "ab") == "ba"
"sample checks passed"

## 6. 为什么是 O(m + n)

设 `m = len(s)`，`n = len(t)`。

- 初始化 `need` 需要扫描 `t`，是 `O(n)`。
- `right` 指针从左到右走一遍，最多移动 `m` 次。
- `left` 指针也只从左到右走，不会回退，最多移动 `m` 次。
- 每次移动只更新一个字符计数和一个标量 `missing`。

所以总时间复杂度是：

$$
O(n) + O(m) + O(m) = O(m+n)
$$

空间复杂度取决于字符集大小。题目说是英文字母，所以可以视作 `O(1)`；如果按一般字符表理解，就是 `O(|alphabet|)`。

In [ ]:
def pointer_move_bound(m, n):
    return {
        "build_need": n,
        "right_moves_at_most": m,
        "left_moves_at_most": m,
        "total_linear_bound": n + 2 * m,
    }

pointer_move_bound(len("ADOBECODEBANC"), len("ABC"))

## 7. 这个题型如何举一反三

看到下面特征时，可以优先考虑可变长滑动窗口：

1. 目标是连续子串或连续子数组。
2. 要找最短或最长区间。
3. 区间是否达标能随着右端扩张呈现单调性。
4. 达标后可以通过移动左端来寻找更优区间。
5. 区间状态可以用计数表、和、不同元素个数等增量维护。

常见变体：

- 最长无重复字符子串：窗口内每个字符计数不超过 1。
- 找所有字母异位词：固定长度窗口 + 计数表匹配。
- 和至少为 K 的最短子数组：窗口和满足阈值后收缩左端，前提是元素非负。
- 至多包含 K 种不同字符的最长子串：用不同字符数控制窗口合法性。

## 8. 复盘问题

写完后，用自己的话回答：

1. 为什么 `t` 不能只建模成集合，而要建模成计数表？
2. 为什么固定 `left` 时，`right` 达标后继续右移仍然达标？
3. 为什么达标后要用 `while` 收缩，而不是只收缩一次？
4. `missing` 表示的是“缺少的字符种类数”还是“缺少的字符总数”？
5. 为什么两个指针各走一遍就够了，不需要回退？